# Data cleaning before and after translation

In [ ]:
import os
import re
import pandas
import unicodedata

from langdetect import detect
from bs4 import BeautifulSoup

In [ ]:
# Basic Cleaning before translation

def bs_process(text):
    #  Remove non-ASCII characters
    text = BeautifulSoup(text.replace("<br>", "\n").replace("<sup>", "").replace("</sup>", ""), "html.parser").get_text(
        separator=". ", strip=True)
    text = (unicodedata.normalize('NFKD', text)
            .encode('ascii', 'ignore')
            .decode('utf-8', 'ignore')
            # .lower() # lower case lowers the sparsity of the data
            )
    # remove numbers
    # text = re.sub(r'\d+', '', str(text))
    # text = re.sub(r'[^A-Za-z0-9|]+', ' ', text)
    # text = re.sub(r'[^A-Za-z0-9]+', ' ', text)
    text = re.sub(r'\n', ' ', text)

    # # removing salutaions (if any)
    # text = re.sub("Mr\.", 'Mr', str(text))
    # text = re.sub("Mrs\.", 'Mrs', str(text))
    #
    # text = re.sub(r"what's", "what is ", text)
    # text = re.sub(r'won\'t', 'will not', text)
    # text = text.replace('(ap)', '')
    # text = re.sub(r"\'s", " is ", text)
    # text = re.sub(r'(\w+)\'s', '\g<1> is', text)
    # text = re.sub(r'(\w+)\'ve', '\g<1> have', text)
    # text = re.sub(r"can't", "cannot ", text)
    # text = re.sub(r"n't", " not ", text)
    # text = re.sub(r"i'm", "i am ", text)
    # text = re.sub(r"\'re", " are ", text)
    # text = re.sub(r"\'d", " would ", text)
    # text = re.sub(r"\'ll", " will ", text)

    # substitute multiple whitespace with single whitespace
    # Also, removes leading and trailing whitespaces
    # text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    # removing any reference to outside text
    # text = re.sub(r"\\", "", text)
    # text = re.sub(r"\'", "", text)
    # text = re.sub(r"\"", "", text)

    text = text.strip()

    # text = re.sub(r'([a-z])([A-Z])', r'\1\. \2', text)  # before lower case
    # normalization 2: lower case
    # text = text.lower()
    # # normalization 3: "&gt", "&lt"
    # text = re.sub(r'&gt|&lt', ' ', text)
    # # normalization 4: letter repetition (if more than 2)
    # text = re.sub(r'([a-z])\1{2,}', r'\1', text)
    # # normalization 5: non-word repetition (if more than 1)
    # text = re.sub(r'([\W+])\1{1,}', r'\1', text)
    # # normalization 6: string * as delimiter
    # text = re.sub(r'\*|\W\*|\*\W', '. ', text)
    # # normalization 7: stuff in parenthesis, assumed to be less informal
    # text = re.sub(r'\(.*?\)', '. ', text)
    # # normalization 8: xxx[?!]. -- > xxx.
    # text = re.sub(r'\W+?\.', '.', text)
    # # normalization 9: [.?!] --> [.?!] xxx
    # text = re.sub(r'(\.|\?|!)(\w)', r'\1 \2', text)
    # # normalization 10: ' ing ', noise text
    # # s = re.sub(r' ing ', ' ', s)
    # # # normalization 11: noise text
    # # s = re.sub(r'product received for free[.| ]', ' ', s)
    # # normalization 12: phrase repetition
    # text = re.sub(r'(.{2,}?)\1{1,}', r'\1', text)
    return text

In [ ]:
file = '/path/to/raw/data' #  (raw data)
ip_df = pandas.read_csv(file, delimiter=";")
ip_df.head()

In [ ]:
ip_df['title_clean'] = ip_df['title'].astype(str).apply(bs_process)
ip_df['content_clean'] = ip_df['content'].astype(str).apply(bs_process)
ip_df[["title_clean", "content_clean"]].head()

In [ ]:
# Refer this path (/translate/) scripts for bulk translation  and them below

In [ ]:
#Combining output translated pickle files into singel df
translated_data_dir = "/path/to/translated.pkl"
all_files = [fn for fn in os.listdir(translated_data_dir) if ".pkl" in fn]

print(f"Total files: {len(all_files)}")
all_data = []
for fn in all_files:
    all_data.append(pandas.read_pickle(os.path.join(translated_data_dir, fn)))

all_translated_data_df = pandas.concat(all_data)
all_translated_data_df.head()

In [ ]:
#language detection
def detect_language(text):
    try:
        return detect(text)
    except:
        return 'unknown'

all_translated_data_df['content_clean_translated_language'] = all_translated_data_df['content_clean_translated'].apply(detect_language) 
all_translated_data_df["content_clean_translated_language"].value_counts()

In [ ]:
def bs_process(text):
    #  Remove non-ASCII characters
    text = BeautifulSoup(text.replace("<br>", "\n").replace("<sup>", "").replace("</sup>", ""), "html.parser").get_text(
        separator=". ", strip=True)
    text = (unicodedata.normalize('NFKD', text)
            .encode('ascii', 'ignore')
            .decode('utf-8', 'ignore')
            # .lower() # lower case lowers the sparsity of the data
            )
# removing any reference to outside text
    text = re.sub(r"\\", "", text)
    text = re.sub(r"\'", "", text)
    text = re.sub(r"\"", "", text)
    text = text.lower()
    # # normalization 4: letter repetition (if more than 2)
    text = re.sub(r'([a-z])\1{2,}', r'\1', text)
    # # normalization 5: non-word repetition (if more than 1)
    text = re.sub(r'([\W+])\1{1,}', r'\1', text)
    # replace * with ' '
    text = re.sub(r'\*|\W\*|\*\W', '', text)
    #text=re.sub(r'[^\w\s]', '', text) # removes this set of symbols [!”#$%&’()*+,-./:;<=>?@[]^_`{|}~]:
    #text=re.sub(r'[^\w\s]','', text) # remove not words and not spaces
    text=re.sub(r'http\S+', '', text) # URL removal
    #Newlines, spaces and tabs removal
    text=re.sub('[\t\n\r\f ]+', ' ', text)
    text=re.sub('[^\S]+', ' ', text)
    text=re.sub('\s+', ' ', text)
    return text

In [ ]:
def remove_urls(text):
	"""
	Return :- String without URLs
	input :- String
	Output :- String
	"""
	url_pattern = r'https?://\S+|www\.\S+'
	without_urls = re.sub(pattern=url_pattern, repl=' ', string=text)
	return without_urls

In [ ]:
ip_df_english_translated['content_translated_processed'] = ip_df_english_translated['content_clean_translated'].astype(str).apply(bs_process)
print("Bs_process is successfully loaded")
ip_df_english_translated['content_translated_processed'] = ip_df_english_translated['content_clean_translated'].astype(str).apply(remove_urls)
print("remove_url is successfully loaded")

In [ ]:
# Remove Punctuation from a String with Translate
string_pun ='!"#$%&\'()*+-/:<=>@[\\]^_`{|}~'
string_punctuation=str.maketrans('','',string_pun)
ip_df_english_translated['content_translated_processed'] = ip_df_english_translated['content_translated_processed'].str.translate(string_punctuation)
ip_df_english_translated['content_translated_processed'][1]

In [ ]:
ip_df_english_translated['content_clean'] = ip_df_english_translated['content_clean'].astype(str).apply(bs_process)
print("Bs_process is successfully loaded")
ip_df_english_translated['content_clean'] = ip_df_english_translated['content_clean'].astype(str).apply(remove_urls)
print("remove_url is successfully loaded")
# Remove Punctuation from a String with Translate
string_pun ='!"#$%&\'()*+-/:<=>@[\\]^_`{|}~'
string_punctuation=str.maketrans('', '', string_pun)
ip_df_english_translated['content_clean'] = ip_df_english_translated['content_clean'].str.translate(string_punctuation)
print("remove punctuation successfull")

In [ ]:
ip_df_english_translated.to_pickle('/path/tosave.pkl')